# Lab 3 — Autoencoder convolucional para MagnaTagATune (V4)

**Machine Learning for Sound and Music — UPF**

Adaptación del notebook docente `06_01_autoencoders.ipynb` al dataset MagnaTagATune (MTAT).

## Experimentos

- **Baseline:** arquitectura `AudioAutoencoder` de referencia, MSE y bottleneck de 512.
- **Mejorado:** misma estructura general y misma compresión, con convoluciones adicionales y pérdida perceptual multiescala STFT del notebook docente.

Se utiliza el primer segundo de cada audio, mono y a 16 kHz, como en el preprocesamiento del notebook de referencia. La tasa de compresión es 16000/512 = 31,25:1.

Los datos preparados se almacenan en `MyDrive/MLSM/Lab 3/data/prepared`. Los resultados se guardan en `MyDrive/MLSM/Lab 3/results_v4`.

**Importante:** si los arrays preparados ya existen, el flujo normal los carga directamente. No vuelve a recorrer, validar ni decodificar los MP3. Las opciones de comprobación o preparación forzada requieren confirmación explícita.

## 1. Dependencias

No reinstalamos PyTorch para evitar alterar la configuración CUDA de Colab. Se instalan las librerías auxiliares, incluyendo UMAP para la proyección final.

In [ ]:
# FFmpeg permite decodificar archivos MP3 cuando sea necesario preparar los datos.
!apt-get -qq update
!apt-get -qq install -y ffmpeg

!pip -q install librosa==0.10.2.post1 scikit-learn matplotlib pandas tqdm umap-learn

## 2. Importaciones, Google Drive y rutas

La carpeta `prepared` es la ubicación única de los arrays que consumen los modelos. La carga normal no verifica firmas ni vuelve a procesar los archivos: si los ficheros preparados están presentes, se leen directamente.

In [ ]:
import os
import json
import math
import random
import shutil
import subprocess
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import umap

from tqdm.auto import tqdm
from IPython.display import Audio, display

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans

from google.colab import drive
drive.mount("/content/drive")

BASE_DIR = Path("/content/drive/MyDrive/MLSM/Lab 3")
DATA_ROOT = BASE_DIR / "data"
DOWNLOAD_DIR = DATA_ROOT / "downloads"
AUDIO_DIR = DATA_ROOT / "audio"
PREPARED_DIR = DATA_ROOT / "prepared"
RESULTS_ROOT = BASE_DIR / "results_v4"

for folder in (DATA_ROOT, DOWNLOAD_DIR, AUDIO_DIR, PREPARED_DIR, RESULTS_ROOT):
    folder.mkdir(parents=True, exist_ok=True)

TRAIN_CACHE = PREPARED_DIR / "train_audio.npy"
VAL_CACHE = PREPARED_DIR / "val_audio.npy"
SPLIT_FILE = PREPARED_DIR / "split_paths.json"
PREPROCESSING_FILE = PREPARED_DIR / "preprocessing.json"

print("Datos preparados:", PREPARED_DIR)
print("Resultados V4:", RESULTS_ROOT)

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Parámetros fijos del preprocesamiento y del ejercicio.
SAMPLE_RATE = 16_000
SEGMENT_SECONDS = 1
INPUT_LENGTH = SAMPLE_RATE * SEGMENT_SECONDS
BOTTLENECK_SIZE = 512
COMPRESSION_RATE = INPUT_LENGTH / BOTTLENECK_SIZE

print(f"Frecuencia: {SAMPLE_RATE} Hz | muestras: {INPUT_LENGTH}")
print(f"Bottleneck: {BOTTLENECK_SIZE} | compresión: {COMPRESSION_RATE:.2f}:1")

## 3. Carga de los datos preparados

### Comportamiento normal

Si existen `train_audio.npy` y `val_audio.npy` en `data/prepared`, se cargan directamente con memoria mapeada. No se comprueba la integridad de cada MP3, no se compara un manifiesto y no se vuelven a preparar los segmentos.

### Acciones opcionales

- `usar`: comportamiento recomendado; carga los arrays existentes.
- `comprobar`: vuelve a localizar y validar los MP3 y comprueba las matrices. Puede tardar.
- `preparar`: fuerza la reconstrucción de los arrays. Puede tardar bastante y sobrescribe los arrays preparados tras confirmación.

Si todavía no existen los arrays, el notebook realizará la preparación inicial.

In [ ]:
# Cambia a "comprobar" o "preparar" solo si realmente quieres repetir esas operaciones.
DATA_ACTION = "usar"  # opciones: "usar", "comprobar", "preparar"

def confirm_long_operation(message):
    answer = input(f"{message}\nEscribe SI para continuar: ").strip()
    return answer == "SI"

def download_and_extract_mtat():
    PARTS = ["mp3.zip.001", "mp3.zip.002", "mp3.zip.003"]
    BASE_URL = "https://mirg.city.ac.uk/datasets/magnatagatune"

    for part in PARTS:
        destination = DOWNLOAD_DIR / part
        if not destination.exists() or destination.stat().st_size == 0:
            print(f"Descargando {part}...")
            subprocess.run(
                ["wget", "-c", f"{BASE_URL}/{part}", "-P", str(DOWNLOAD_DIR)],
                check=True
            )
        else:
            print(f"Parte ya descargada: {part}")

    merged_archive = DOWNLOAD_DIR / "mp3.zip"
    expected_size = sum((DOWNLOAD_DIR / part).stat().st_size for part in PARTS)

    if not merged_archive.exists() or merged_archive.stat().st_size != expected_size:
        print("Combinando las partes del archivo...")
        temporary_archive = DOWNLOAD_DIR / "mp3.zip.tmp"
        with open(temporary_archive, "wb") as output_file:
            for part in PARTS:
                with open(DOWNLOAD_DIR / part, "rb") as input_file:
                    shutil.copyfileobj(input_file, output_file)
        subprocess.run(["unzip", "-tq", str(temporary_archive)], check=True)
        temporary_archive.replace(merged_archive)
    else:
        print("Archivo ZIP combinado ya disponible.")

    marker = AUDIO_DIR / ".extraction_complete"
    mp3_count = sum(1 for _ in AUDIO_DIR.rglob("*.mp3"))
    extraction_done = False
    if marker.exists():
        try:
            extraction_done = int(marker.read_text().strip()) == mp3_count and mp3_count > 0
        except (ValueError, OSError):
            extraction_done = False

    if not extraction_done:
        print("Extrayendo MTAT...")
        subprocess.run(
            ["unzip", "-q", "-o", str(merged_archive), "-d", str(AUDIO_DIR)],
            check=True
        )
        mp3_count = sum(1 for _ in AUDIO_DIR.rglob("*.mp3"))
        if mp3_count == 0:
            raise RuntimeError("La extracción no produjo archivos MP3.")
        marker.write_text(str(mp3_count))
    else:
        print(f"Extracción ya disponible ({mp3_count:,} MP3).")

def prepare_arrays(force=False):
    all_paths = sorted(AUDIO_DIR.rglob("*.mp3"))
    if not all_paths:
        raise RuntimeError("No se encontraron MP3 en la carpeta de audio.")

    if force:
        print("Se comprobarán de nuevo todos los MP3. Esta operación puede tardar.")
    valid_paths = []
    for audio_path in tqdm(all_paths, desc="Comprobando duración y lectura"):
        try:
            duration = librosa.get_duration(path=str(audio_path))
            if duration >= SEGMENT_SECONDS:
                valid_paths.append(audio_path)
        except Exception:
            continue

    if not valid_paths:
        raise RuntimeError("No se encontraron audios válidos de al menos un segundo.")

    train_paths, val_paths = train_test_split(
        valid_paths, test_size=0.20, random_state=SEED, shuffle=True
    )

    def write_matrix(paths, destination):
        temporary = destination.with_suffix(".tmp.npy")
        matrix = np.lib.format.open_memmap(
            temporary, mode="w+", dtype=np.float32,
            shape=(len(paths), INPUT_LENGTH)
        )
        for index, audio_path in enumerate(tqdm(paths, desc=f"Preparando {destination.name}")):
            audio, _ = librosa.load(
                path=str(audio_path), sr=SAMPLE_RATE, mono=True
            )
            if len(audio) < INPUT_LENGTH:
                audio = np.pad(audio, (0, INPUT_LENGTH - len(audio)))
            segment = audio[:INPUT_LENGTH].astype(np.float32)
            peak = np.max(np.abs(segment))
            if peak > 0:
                segment = segment / peak
            matrix[index] = segment
        matrix.flush()
        del matrix
        temporary.replace(destination)

    write_matrix(train_paths, TRAIN_CACHE)
    write_matrix(val_paths, VAL_CACHE)

    split_data = {
        "train_paths": [str(p.relative_to(AUDIO_DIR)) for p in train_paths],
        "val_paths": [str(p.relative_to(AUDIO_DIR)) for p in val_paths]
    }
    SPLIT_FILE.write_text(json.dumps(split_data, indent=2), encoding="utf-8")
    preprocessing = {
        "sample_rate": SAMPLE_RATE,
        "segment_seconds": SEGMENT_SECONDS,
        "input_length": INPUT_LENGTH,
        "crop": "first_segment",
        "normalization": "peak_1",
        "split_seed": SEED,
        "split_ratio": 0.8
    }
    PREPROCESSING_FILE.write_text(
        json.dumps(preprocessing, indent=2), encoding="utf-8"
    )
    print(f"Arrays guardados en {PREPARED_DIR}")

prepared_exists = TRAIN_CACHE.exists() and VAL_CACHE.exists()

if DATA_ACTION not in {"usar", "comprobar", "preparar"}:
    raise ValueError("DATA_ACTION debe ser 'usar', 'comprobar' o 'preparar'.")

if DATA_ACTION == "usar" and prepared_exists:
    print("Se reutilizan directamente los arrays preparados de Drive.")
elif DATA_ACTION == "comprobar":
    if prepared_exists:
        if confirm_long_operation("Vas a volver a localizar y comprobar todos los MP3. Puede tardar bastante."):
            if not any(AUDIO_DIR.rglob("*.mp3")):
                download_and_extract_mtat()
            prepare_arrays(force=True)
        else:
            print("Comprobación cancelada. Se usarán los arrays existentes.")
    else:
        print("No hay arrays preparados. Se realizará la preparación inicial.")
        if not any(AUDIO_DIR.rglob("*.mp3")):
            download_and_extract_mtat()
        prepare_arrays(force=False)
elif DATA_ACTION == "preparar":
    if prepared_exists:
        if confirm_long_operation("Vas a sobrescribir los arrays preparados. La operación puede tardar bastante."):
            if not any(AUDIO_DIR.rglob("*.mp3")):
                download_and_extract_mtat()
            prepare_arrays(force=True)
        else:
            print("Preparación cancelada. Se conservarán los arrays existentes.")
    else:
        print("No hay arrays preparados. Se realizará la preparación inicial.")
        if not any(AUDIO_DIR.rglob("*.mp3")):
            download_and_extract_mtat()
        prepare_arrays(force=False)
elif not prepared_exists:
    print("No existen arrays preparados. Se realizará la preparación inicial.")
    if not any(AUDIO_DIR.rglob("*.mp3")):
        download_and_extract_mtat()
    prepare_arrays(force=False)

if not (TRAIN_CACHE.exists() and VAL_CACHE.exists()):
    raise RuntimeError("No hay arrays preparados disponibles para entrenar.")

# Carga directa; no se hace una comprobación exhaustiva en el flujo normal.
train_audio = np.load(TRAIN_CACHE, mmap_mode="r")
val_audio = np.load(VAL_CACHE, mmap_mode="r")
print("Train:", train_audio.shape, train_audio.dtype)
print("Validation:", val_audio.shape, val_audio.dtype)

## 4. Dataset y DataLoaders

El Dataset lee los arrays `.npy` directamente desde Drive mediante memory mapping. Los MP3 no intervienen en el entrenamiento.

In [ ]:
class PreparedAudioDataset(Dataset):
    def __init__(self, array):
        self.array = array

    def __len__(self):
        return self.array.shape[0]

    def __getitem__(self, index):
        audio = np.array(self.array[index], dtype=np.float32, copy=True)
        return torch.from_numpy(audio)

train_dataset = PreparedAudioDataset(train_audio)
val_dataset = PreparedAudioDataset(val_audio)

def make_train_loader(batch_size, seed=SEED):
    generator = torch.Generator()
    generator.manual_seed(seed)
    return DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=0, pin_memory=torch.cuda.is_available(),
        generator=generator
    )

def make_val_loader(batch_size):
    return DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        num_workers=0, pin_memory=torch.cuda.is_available()
    )

print("Ejemplos train:", len(train_dataset))
print("Ejemplos validation:", len(val_dataset))

## 5. Arquitectura compartida

La clase implementa la arquitectura del notebook docente. El argumento `improved=True` activa únicamente las convoluciones adicionales del experimento mejorado. El tamaño del bottleneck se recibe explícitamente desde cada bloque de parámetros.

In [ ]:
class AudioAutoencoder(nn.Module):
    def __init__(
        self,
        input_length=16000,
        bottleneck_size=512,
        kernel_size=5,
        channels=8,
        channel_mult=2,
        downsample=2,
        improved=False
    ):
        super().__init__()
        if kernel_size % 2 != 1:
            raise ValueError("kernel_size debe ser impar.")
        self.improved = improved
        self.input_length = input_length
        self.bottleneck_size = bottleneck_size
        padding = kernel_size // 2

        # Encoder original: Conv1d + BatchNorm + PReLU + MaxPool con índices.
        self.enc_conv1 = nn.Conv1d(1, channels, kernel_size, padding=padding)
        self.enc_norm1 = nn.BatchNorm1d(channels)
        self.enc_act1 = nn.PReLU()
        self.pool1 = nn.MaxPool1d(2, stride=downsample, return_indices=True)

        self.enc_conv2 = nn.Conv1d(channels, channels * channel_mult, kernel_size, padding=padding)
        self.enc_norm2 = nn.BatchNorm1d(channels * channel_mult)
        self.enc_act2 = nn.PReLU()
        self.pool2 = nn.MaxPool1d(2, stride=downsample, return_indices=True)

        self.enc_conv3 = nn.Conv1d(
            channels * channel_mult, channels * channel_mult ** 2,
            kernel_size, padding=padding
        )
        self.enc_norm3 = nn.BatchNorm1d(channels * channel_mult ** 2)
        self.enc_act3 = nn.PReLU()
        self.pool3 = nn.MaxPool1d(2, stride=downsample, return_indices=True)

        encoded_length = input_length // (downsample ** 3)
        encoded_features = encoded_length * channels * channel_mult ** 2
        self.enc_fc = nn.Linear(encoded_features, bottleneck_size)
        self.dec_fc = nn.Linear(bottleneck_size, encoded_features)

        # Capas adicionales del modelo mejorado.
        if improved:
            self.enc_extra1 = nn.Conv1d(channels, channels, kernel_size, padding=padding)
            self.enc_extra2 = nn.Conv1d(channels * channel_mult, channels * channel_mult, kernel_size, padding=padding)
            self.enc_extra3 = nn.Conv1d(channels * channel_mult ** 2, channels * channel_mult ** 2, kernel_size, padding=padding)

        # Decoder original.
        self.unpool3 = nn.MaxUnpool1d(2, stride=downsample)
        self.dec_deconv3 = nn.Conv1d(channels * channel_mult ** 2, channels * channel_mult, kernel_size, padding=padding)
        self.dec_act3 = nn.PReLU()

        self.unpool2 = nn.MaxUnpool1d(2, stride=downsample)
        self.dec_deconv2 = nn.Conv1d(channels * channel_mult, channels, kernel_size, padding=padding)
        self.dec_act2 = nn.PReLU()

        self.unpool1 = nn.MaxUnpool1d(2, stride=downsample)
        self.dec_deconv1 = nn.Conv1d(channels, 1, kernel_size, padding=padding)
        self.dec_tanh = nn.Tanh()

        if improved:
            self.dec_extra3 = nn.Conv1d(channels * channel_mult, channels * channel_mult, kernel_size, padding=padding)
            self.dec_extra2 = nn.Conv1d(channels, channels, kernel_size, padding=padding)

        self.downsample = downsample
        print(f"Compression rate: {input_length / bottleneck_size:.2f}:1")

    def encoder(self, x):
        sizes, indices = [], []

        x = self.enc_act1(self.enc_norm1(self.enc_conv1(x)))
        if self.improved:
            x = self.enc_act1(self.enc_extra1(x))
        sizes.append(x.size())
        x, idx1 = self.pool1(x)
        indices.append(idx1)

        x = self.enc_act2(self.enc_norm2(self.enc_conv2(x)))
        if self.improved:
            x = self.enc_act2(self.enc_extra2(x))
        sizes.append(x.size())
        x, idx2 = self.pool2(x)
        indices.append(idx2)

        x = self.enc_act3(self.enc_norm3(self.enc_conv3(x)))
        if self.improved:
            x = self.enc_act3(self.enc_extra3(x))
        sizes.append(x.size())
        x, idx3 = self.pool3(x)
        indices.append(idx3)

        z = self.enc_fc(x.flatten(start_dim=1))
        return z, indices, sizes

    def decoder(self, z, indices, sizes):
        idx1, idx2, idx3 = indices
        size1, size2, size3 = sizes

        x = self.dec_fc(z).view(size3[0], size3[1], -1)
        x = self.unpool3(x, idx3, output_size=size3)
        x = self.dec_act3(self.dec_deconv3(x))
        if self.improved:
            x = self.dec_act3(self.dec_extra3(x))

        x = self.unpool2(x, idx2, output_size=size2)
        x = self.dec_act2(self.dec_deconv2(x))
        if self.improved:
            x = self.dec_act2(self.dec_extra2(x))

        x = self.unpool1(x, idx1, output_size=size1)
        return self.dec_tanh(self.dec_deconv1(x))

    def forward(self, x):
        z, indices, sizes = self.encoder(x)
        return self.decoder(z, indices, sizes), z

# Prueba de dimensiones y compresión antes del entrenamiento.
for use_improved in (False, True):
    test_model = AudioAutoencoder(
        input_length=INPUT_LENGTH, bottleneck_size=BOTTLENECK_SIZE,
        downsample=2, improved=use_improved
    ).to(DEVICE)
    with torch.no_grad():
        test_x = torch.zeros(2, 1, INPUT_LENGTH, device=DEVICE)
        test_y, test_z = test_model(test_x)
    assert test_y.shape == test_x.shape
    assert test_z.shape == (2, BOTTLENECK_SIZE)
    print("Improved:", use_improved, "| output:", tuple(test_y.shape), "| latent:", tuple(test_z.shape))
    del test_model, test_x, test_y, test_z
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 6. Funciones de pérdida y entrenamiento

El baseline se optimiza con MSE. El mejorado utiliza la pérdida multiescala STFT del notebook de referencia. Para ambos se calcula el MSE de validación y se guarda el checkpoint con el menor valor de esa métrica.

No se comparan directamente las curvas de pérdida de entrenamiento de ambos modelos: MSE y STFT son magnitudes diferentes.

In [ ]:
mse_criterion = nn.MSELoss()

def multi_scale_stft_loss(x, y, scales=(512, 1024, 2048), hop_ratio=0.25):
    # Implementación de la pérdida perceptual del notebook docente.
    loss = 0.0
    x = x.squeeze(1)
    y = y.squeeze(1)
    for n_fft in scales:
        hop = int(n_fft * hop_ratio)
        window = torch.hann_window(n_fft, device=x.device)
        X = torch.stft(x, n_fft=n_fft, hop_length=hop, window=window, return_complex=True)
        Y = torch.stft(y, n_fft=n_fft, hop_length=hop, window=window, return_complex=True)
        loss = loss + torch.mean(torch.abs(X.abs() - Y.abs()))
    return loss / len(scales)

def evaluate_mse(model, loader):
    model.eval()
    squared_error = 0.0
    n_elements = 0
    with torch.no_grad():
        for batch in loader:
            x = batch.unsqueeze(1).to(DEVICE)
            reconstruction, _ = model(x)
            squared_error += torch.sum((reconstruction - x) ** 2).item()
            n_elements += x.numel()
    return squared_error / n_elements

def train_model(model, train_loader, val_loader, epochs, learning_rate,
                loss_type, checkpoint_path, run_name):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history = {"train_loss": [], "val_mse": []}
    best_val_mse = float("inf")
    best_epoch = None

    for epoch in range(epochs):
        model.train()
        train_losses = []
        for batch in tqdm(train_loader, desc=f"{run_name} | epoch {epoch + 1}/{epochs}"):
            x = batch.unsqueeze(1).to(DEVICE)
            optimizer.zero_grad()
            reconstruction, _ = model(x)
            if loss_type == "MSE":
                loss = mse_criterion(reconstruction, x)
            elif loss_type == "multi_scale_STFT":
                loss = multi_scale_stft_loss(x, reconstruction)
            else:
                raise ValueError(f"Loss no reconocida: {loss_type}")
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        train_loss = float(np.mean(train_losses))
        val_mse = evaluate_mse(model, val_loader)
        history["train_loss"].append(train_loss)
        history["val_mse"].append(val_mse)

        print(
            f"{run_name} | epoch {epoch + 1}/{epochs} "
            f"| train {loss_type}: {train_loss:.6f} | validation MSE: {val_mse:.6f}"
        )

        if val_mse < best_val_mse:
            best_val_mse = val_mse
            best_epoch = epoch + 1
            torch.save(
                {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                checkpoint_path
            )

    if best_epoch is None:
        raise RuntimeError("No se ha guardado ningún checkpoint.")

    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE, weights_only=True))
    return model, history, best_val_mse, best_epoch

## 7. Parámetros del baseline

Este bloque es independiente del entrenamiento del baseline. Si no se modifican sus parámetros, el checkpoint existente se reutiliza.

La configuración del baseline se guarda junto al modelo. Si ya hay un checkpoint, el notebook pregunta antes de volver a entrenarlo. El entrenamiento forzado puede tardar.

In [ ]:
# ===== PARÁMETROS BASELINE =====
BASELINE_PARAMS = {
    "input_length": INPUT_LENGTH,
    "bottleneck_size": BOTTLENECK_SIZE,
    "kernel_size": 5,
    "channels": 8,
    "channel_mult": 2,
    "downsample": 2,
    "batch_size": 32,
    "epochs": 20,
    "learning_rate": 1e-3,
    "seed": SEED,
    "loss": "MSE"
}

assert BASELINE_PARAMS["input_length"] / BASELINE_PARAMS["bottleneck_size"] == COMPRESSION_RATE
print(json.dumps(BASELINE_PARAMS, indent=2))

## 8. Entrenamiento o carga del baseline

El baseline se entrena una sola vez por configuración. Si existen pesos e historial compatibles, se cargan. Para repetir el entrenamiento hay que responder `SI` a la confirmación.

In [ ]:
# ===== ENTRENAMIENTO BASELINE =====
baseline_dir = RESULTS_ROOT / "baseline"
baseline_dir.mkdir(parents=True, exist_ok=True)
baseline_checkpoint = baseline_dir / "baseline_best.pt"
baseline_history_path = baseline_dir / "baseline_history.csv"
baseline_summary_path = baseline_dir / "baseline_summary.json"
baseline_config_path = baseline_dir / "baseline_config.json"

baseline_config = dict(BASELINE_PARAMS)
baseline_config["architecture"] = "AudioAutoencoder_reference"
baseline_config["prepared_data"] = str(PREPARED_DIR)

baseline_retrain = False
if baseline_checkpoint.exists() and baseline_history_path.exists() and baseline_config_path.exists():
    try:
        previous_config = json.loads(baseline_config_path.read_text(encoding="utf-8"))
        if previous_config == baseline_config:
            baseline_retrain = confirm_long_operation(
                "Ya existe un baseline con esta configuración. ¿Quieres volver a entrenarlo? Puede tardar bastante."
            )
        else:
            print("La configuración del baseline ha cambiado.")
            baseline_retrain = confirm_long_operation(
                "Se necesita un nuevo entrenamiento para esta configuración. Puede tardar bastante. ¿Quieres continuar?"
            )
    except (OSError, json.JSONDecodeError):
        print("No se han podido leer los metadatos del baseline; se solicitará confirmación.")
        baseline_retrain = confirm_long_operation(
            "No se puede verificar el baseline guardado. ¿Quieres volver a entrenarlo? Puede tardar bastante."
        )
else:
    print("No existe un baseline guardado; se entrenará por primera vez.")
    baseline_retrain = True

baseline_model = AudioAutoencoder(
    input_length=BASELINE_PARAMS["input_length"],
    bottleneck_size=BASELINE_PARAMS["bottleneck_size"],
    kernel_size=BASELINE_PARAMS["kernel_size"],
    channels=BASELINE_PARAMS["channels"],
    channel_mult=BASELINE_PARAMS["channel_mult"],
    downsample=BASELINE_PARAMS["downsample"],
    improved=False
).to(DEVICE)

if not baseline_retrain:
    print("Se carga el baseline existente; no se repite el entrenamiento.")
    baseline_model.load_state_dict(
        torch.load(baseline_checkpoint, map_location=DEVICE, weights_only=True)
    )
    baseline_history_df = pd.read_csv(baseline_history_path)
    baseline_history = {
        "train_loss": baseline_history_df["train_loss"].tolist(),
        "val_mse": baseline_history_df["val_mse"].tolist()
    }
    baseline_summary = json.loads(baseline_summary_path.read_text(encoding="utf-8"))
    baseline_best_mse = float(baseline_summary["best_validation_mse"])
    baseline_best_epoch = int(baseline_summary["best_epoch"])
else:
    set_seed(BASELINE_PARAMS["seed"])
    baseline_train_loader = make_train_loader(
        BASELINE_PARAMS["batch_size"], BASELINE_PARAMS["seed"]
    )
    baseline_val_loader = make_val_loader(BASELINE_PARAMS["batch_size"])
    baseline_model, baseline_history, baseline_best_mse, baseline_best_epoch = train_model(
        baseline_model, baseline_train_loader, baseline_val_loader,
        epochs=BASELINE_PARAMS["epochs"],
        learning_rate=BASELINE_PARAMS["learning_rate"],
        loss_type=BASELINE_PARAMS["loss"],
        checkpoint_path=baseline_checkpoint,
        run_name="Baseline"
    )
    pd.DataFrame(baseline_history).to_csv(baseline_history_path, index_label="epoch")
    baseline_summary = {
        "best_validation_mse": baseline_best_mse,
        "best_epoch": baseline_best_epoch,
        "trainable_parameters": sum(p.numel() for p in baseline_model.parameters() if p.requires_grad)
    }
    baseline_summary_path.write_text(json.dumps(baseline_summary, indent=2), encoding="utf-8")
    baseline_config_path.write_text(json.dumps(baseline_config, indent=2), encoding="utf-8")

baseline_parameters = sum(p.numel() for p in baseline_model.parameters() if p.requires_grad)
print("Baseline best validation MSE:", baseline_best_mse)
print("Baseline best epoch:", baseline_best_epoch)
print("Baseline trainable parameters:", baseline_parameters)
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 9. Parámetros del modelo mejorado

Este bloque es independiente del entrenamiento. Para probar otra configuración, modifica únicamente este diccionario y ejecuta después la celda de entrenamiento del modelo mejorado.

Se mantienen fijos el tamaño de entrada, el bottleneck y el downsampling del baseline. Las variaciones permitidas para este experimento son las capas convolucionales adicionales y los parámetros de entrenamiento de la pérdida STFT, siguiendo las propuestas del ejercicio.

Cada configuración genera una subcarpeta propia dentro de `results_v4/improved`, identificada mediante un hash de sus parámetros. Dentro se guarda `config.json` con todos los parámetros usados.

In [ ]:
# ===== PARÁMETROS MODELO MEJORADO =====
IMPROVED_PARAMS = {
    # Se mantienen iguales al baseline para conservar la compresión.
    "input_length": INPUT_LENGTH,
    "bottleneck_size": BOTTLENECK_SIZE,
    "kernel_size": 5,
    "channels": 8,
    "channel_mult": 2,
    "downsample": 2,

    # Capas convolucionales adicionales: cambio arquitectónico permitido.
    "additional_encoder_convolutions": 3,
    "additional_decoder_convolutions": 2,

    # Entrenamiento según el notebook docente.
    "batch_size": 32,
    "epochs": 20,
    "learning_rate": 1e-3,
    "seed": SEED,
    "loss": "multi_scale_STFT",
    "stft_scales": [512, 1024, 2048],
    "stft_hop_ratio": 0.25
}

assert IMPROVED_PARAMS["additional_encoder_convolutions"] == 3
assert IMPROVED_PARAMS["additional_decoder_convolutions"] == 2
assert IMPROVED_PARAMS["input_length"] == BASELINE_PARAMS["input_length"]
assert IMPROVED_PARAMS["bottleneck_size"] == BASELINE_PARAMS["bottleneck_size"]
assert IMPROVED_PARAMS["downsample"] == BASELINE_PARAMS["downsample"]
assert IMPROVED_PARAMS["input_length"] / IMPROVED_PARAMS["bottleneck_size"] == COMPRESSION_RATE

print(json.dumps(IMPROVED_PARAMS, indent=2))
print(f"Compresión mantenida: {COMPRESSION_RATE:.2f}:1")

## 10. Entrenamiento del modelo mejorado

Esta celda siempre ejecuta el entrenamiento del modelo mejorado con los parámetros definidos justo antes. La carpeta de salida depende del hash de la configuración: si cambias un parámetro, se crea una nueva subcarpeta y se conservan los resultados anteriores.

Se guardan los parámetros completos, el historial, el mejor checkpoint, un resumen y el identificador de la configuración.

In [ ]:
# ===== ENTRENAMIENTO MODELO MEJORADO =====
set_seed(IMPROVED_PARAMS["seed"])

improved_config = dict(IMPROVED_PARAMS)
improved_config["architecture"] = "AudioAutoencoder_plus_extra_convolutions"
improved_config["prepared_data"] = str(PREPARED_DIR)
improved_config["baseline_compression_rate"] = COMPRESSION_RATE

config_json = json.dumps(improved_config, sort_keys=True, ensure_ascii=False)
config_hash = hashlib.sha256(config_json.encode("utf-8")).hexdigest()[:12]
improved_run_dir = RESULTS_ROOT / "improved" / f"config_{config_hash}"
improved_run_dir.mkdir(parents=True, exist_ok=True)

# Documento legible y completo de la configuración de este entrenamiento.
(improved_run_dir / "config.json").write_text(
    json.dumps(improved_config, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

improved_checkpoint = improved_run_dir / "improved_best.pt"
improved_history_path = improved_run_dir / "improved_history.csv"
improved_summary_path = improved_run_dir / "summary.json"

improved_model = AudioAutoencoder(
    input_length=IMPROVED_PARAMS["input_length"],
    bottleneck_size=IMPROVED_PARAMS["bottleneck_size"],
    kernel_size=IMPROVED_PARAMS["kernel_size"],
    channels=IMPROVED_PARAMS["channels"],
    channel_mult=IMPROVED_PARAMS["channel_mult"],
    downsample=IMPROVED_PARAMS["downsample"],
    improved=True
).to(DEVICE)

# Las escalas STFT configuradas se aplican explícitamente en este experimento.
def improved_loss_function(x, y):
    return multi_scale_stft_loss(
        x, y,
        scales=tuple(IMPROVED_PARAMS["stft_scales"]),
        hop_ratio=IMPROVED_PARAMS["stft_hop_ratio"]
    )

# Entrenamiento con la pérdida perceptual indicada en el notebook docente.
def train_improved_model(model, train_loader, val_loader):
    optimizer = torch.optim.Adam(
        model.parameters(), lr=IMPROVED_PARAMS["learning_rate"]
    )
    history = {"train_loss": [], "val_mse": []}
    best_val_mse = float("inf")
    best_epoch = None

    for epoch in range(IMPROVED_PARAMS["epochs"]):
        model.train()
        losses = []
        for batch in tqdm(
            train_loader,
            desc=f"Improved | epoch {epoch + 1}/{IMPROVED_PARAMS['epochs']}"
        ):
            x = batch.unsqueeze(1).to(DEVICE)
            optimizer.zero_grad()
            reconstruction, _ = model(x)
            loss = improved_loss_function(x, reconstruction)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

        train_loss = float(np.mean(losses))
        val_mse = evaluate_mse(model, val_loader)
        history["train_loss"].append(train_loss)
        history["val_mse"].append(val_mse)

        print(
            f"Improved | epoch {epoch + 1}/{IMPROVED_PARAMS['epochs']} "
            f"| train STFT: {train_loss:.6f} | validation MSE: {val_mse:.6f}"
        )

        if val_mse < best_val_mse:
            best_val_mse = val_mse
            best_epoch = epoch + 1
            torch.save(
                {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                improved_checkpoint
            )

    if best_epoch is None:
        raise RuntimeError("No se ha guardado ningún checkpoint del modelo mejorado.")
    model.load_state_dict(
        torch.load(improved_checkpoint, map_location=DEVICE, weights_only=True)
    )
    return model, history, best_val_mse, best_epoch

improved_train_loader = make_train_loader(
    IMPROVED_PARAMS["batch_size"], IMPROVED_PARAMS["seed"]
)
improved_val_loader = make_val_loader(IMPROVED_PARAMS["batch_size"])

improved_model, improved_history, improved_best_mse, improved_best_epoch = train_improved_model(
    improved_model, improved_train_loader, improved_val_loader
)

improved_parameters = sum(
    p.numel() for p in improved_model.parameters() if p.requires_grad
)
pd.DataFrame(improved_history).to_csv(improved_history_path, index_label="epoch")

improved_summary = {
    "config_hash": config_hash,
    "best_validation_mse": improved_best_mse,
    "best_epoch": improved_best_epoch,
    "trainable_parameters": improved_parameters,
    "checkpoint": str(improved_checkpoint)
}
improved_summary_path.write_text(
    json.dumps(improved_summary, indent=2), encoding="utf-8"
)

print("Resultados de esta configuración:", improved_run_dir)
print("Mejor validation MSE:", improved_best_mse)
print("Mejor época:", improved_best_epoch)
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 11. Comparación de los modelos

La comparación principal se hace mediante el menor MSE de validación de cada modelo. La pérdida de entrenamiento del baseline es MSE y la del mejorado es STFT; por tanto, no deben interpretarse como una misma magnitud.

In [ ]:
comparison = pd.DataFrame({
    "Model": ["Baseline", "Improved"],
    "Best validation MSE": [baseline_best_mse, improved_best_mse],
    "Best epoch": [baseline_best_epoch, improved_best_epoch],
    "Trainable parameters": [baseline_parameters, improved_parameters]
})
display(comparison)

relative_improvement = (
    (baseline_best_mse - improved_best_mse) / baseline_best_mse
) * 100
print(f"Variación relativa del validation MSE: {relative_improvement:.2f}%")
if improved_best_mse < baseline_best_mse:
    print("El modelo mejorado obtiene menor MSE de validación.")
elif improved_best_mse > baseline_best_mse:
    print("El baseline obtiene menor MSE de validación.")
else:
    print("Ambos modelos obtienen el mismo MSE de validación.")

comparison.to_csv(improved_run_dir / "model_comparison.csv", index=False)
comparison.to_csv(RESULTS_ROOT / "latest_model_comparison.csv", index=False)

## 12. Curvas de entrenamiento y validación

Se muestran las pérdidas de entrenamiento en paneles separados porque sus escalas y definiciones son diferentes. El MSE de validación sí es común a ambos modelos y se representa conjuntamente.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(np.arange(1, len(baseline_history["train_loss"]) + 1),
             baseline_history["train_loss"], label="Baseline train MSE")
axes[0].set_title("Baseline — training MSE")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("MSE")
axes[0].grid(True, alpha=0.3)

axes[1].plot(np.arange(1, len(improved_history["train_loss"]) + 1),
             improved_history["train_loss"], label="Improved train STFT")
axes[1].set_title("Improved — training STFT loss")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("STFT loss")
axes[1].grid(True, alpha=0.3)

axes[2].plot(np.arange(1, len(baseline_history["val_mse"]) + 1),
             baseline_history["val_mse"], label="Baseline")
axes[2].plot(np.arange(1, len(improved_history["val_mse"]) + 1),
             improved_history["val_mse"], label="Improved")
axes[2].set_title("Validation MSE")
axes[2].set_xlabel("Época")
axes[2].set_ylabel("MSE")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(improved_run_dir / "training_curves.png", dpi=200, bbox_inches="tight")
plt.show()

## 13. Escucha y visualización de reconstrucciones

Se usa el mismo ejemplo del conjunto de validación para ambos modelos.

In [ ]:
example_batch = next(iter(make_val_loader(8)))
example = example_batch[0].unsqueeze(0).unsqueeze(1).to(DEVICE)

baseline_model.eval()
improved_model.eval()
with torch.no_grad():
    baseline_reconstruction, _ = baseline_model(example)
    improved_reconstruction, _ = improved_model(example)

original_audio = example[0, 0].cpu().numpy()
baseline_audio = baseline_reconstruction[0, 0].cpu().numpy()
improved_audio = improved_reconstruction[0, 0].cpu().numpy()

display(Audio(original_audio, rate=SAMPLE_RATE))
display(Audio(baseline_audio, rate=SAMPLE_RATE))
display(Audio(improved_audio, rate=SAMPLE_RATE))

time_axis = np.arange(INPUT_LENGTH) / SAMPLE_RATE
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
axes[0].plot(time_axis, original_audio)
axes[0].set_title("Audio original")
axes[1].plot(time_axis, baseline_audio)
axes[1].set_title("Reconstrucción baseline")
axes[2].plot(time_axis, improved_audio)
axes[2].set_title("Reconstrucción mejorada")
axes[2].set_xlabel("Tiempo (segundos)")
for ax in axes:
    ax.set_ylabel("Amplitud")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(improved_run_dir / "audio_reconstruction_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

import soundfile as sf
sf.write(improved_run_dir / "validation_original.wav", original_audio, SAMPLE_RATE)
sf.write(improved_run_dir / "validation_baseline_reconstruction.wav", baseline_audio, SAMPLE_RATE)
sf.write(improved_run_dir / "validation_improved_reconstruction.wav", improved_audio, SAMPLE_RATE)

## 14. Proyección UMAP del espacio latente

Se extraen los embeddings del conjunto de validación usando el modelo mejorado, igual que en la última sección del notebook docente.

MTAT no proporciona en este flujo las etiquetas instrumentales de TinySOL que aparecen en el ejemplo docente. Para no inventar etiquetas, se colorea la proyección con clusters obtenidos mediante K-Means sobre los embeddings. Estos clusters son agrupaciones exploratorias, no etiquetas semánticas reales.

In [ ]:
# Extraer embeddings del conjunto de validación.
improved_model.eval()
embedding_batches = []
with torch.no_grad():
    for batch in tqdm(make_val_loader(128), desc="Extrayendo embeddings"):
        x = batch.unsqueeze(1).to(DEVICE)
        _, z = improved_model(x)
        embedding_batches.append(z.cpu().numpy())

embeddings = np.concatenate(embedding_batches, axis=0)

# Agrupación exploratoria para colorear la proyección.
n_clusters = min(8, len(embeddings))
cluster_labels = KMeans(
    n_clusters=n_clusters, random_state=SEED, n_init=10
).fit_predict(embeddings)

reducer = umap.UMAP(
    n_components=2, random_state=SEED, n_neighbors=15, min_dist=0.1
)
umap_embeddings = reducer.fit_transform(embeddings)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    umap_embeddings[:, 0], umap_embeddings[:, 1],
    c=cluster_labels, cmap="tab10", s=5
)
plt.colorbar(scatter, label="Cluster K-Means")
plt.title("UMAP of Improved Audio Embeddings — Validation Set")
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.tight_layout()
plt.savefig(improved_run_dir / "umap_latent_space.png", dpi=200, bbox_inches="tight")
plt.show()

np.savez_compressed(
    improved_run_dir / "umap_embeddings.npz",
    embeddings=embeddings,
    umap_embeddings=umap_embeddings,
    cluster_labels=cluster_labels
)

## 15. Archivos generados y reutilización

Estructura principal en Google Drive:

```
MLSM/Lab 3/
├── data/
│   ├── downloads/
│   ├── audio/
│   └── prepared/
│       ├── train_audio.npy
│       ├── val_audio.npy
│       ├── split_paths.json
│       └── preprocessing.json
└── results_v4/
    ├── baseline/
    │   ├── baseline_best.pt
    │   ├── baseline_history.csv
    │   ├── baseline_summary.json
    │   └── baseline_config.json
    └── improved/
        └── config_<hash>/
            ├── config.json
            ├── improved_best.pt
            ├── improved_history.csv
            ├── summary.json
            ├── model_comparison.csv
            ├── training_curves.png
            ├── audio_reconstruction_comparison.png
            └── umap_latent_space.png
```

- **Datos:** la ejecución normal carga directamente los arrays de `data/prepared`. No comprueba de nuevo ni prepara los MP3.
- **Baseline:** se carga desde `results_v4/baseline`. Solo se vuelve a entrenar si no existe, cambia su configuración o se confirma explícitamente.
- **Mejorado:** para probar parámetros distintos, modifica únicamente la celda de parámetros del modelo mejorado y ejecuta su celda de entrenamiento. Cada configuración distinta crea una subcarpeta nueva con un `config.json` que documenta todos los parámetros.
- **UMAP:** la proyección y los embeddings se guardan dentro de la carpeta de la configuración mejorada correspondiente.